In [1]:
# TITULO: Entrenamiento Comparativo - Detector de Placas
import os
from ultralytics import YOLO
from datetime import datetime
import pandas as pd
import matplotlib.pyplot as plt

# Rutas de los datasets generados
YAML_CLEAN = '../../datasets/02_placas/data.yaml'
# YAML_BASELINE = '../../datasets/02_placas_baseline/data.yaml'

# Ruta de salida de modelos
MODELS_DIR = '../../../../models/02_placas'

# Fecha para versionado
DATE_STR = datetime.now().strftime('%Y%m%d')

print("Configuracion lista.")

Configuracion lista.


In [2]:
# Función para obtener el optimizador real cuando se usa 'auto'
def obtener_optimizador_real(modelo):
    """
    Recupera el nombre real del optimizador cuando la configuración es 'auto'.
    """
    try:
        # 1. Si el entrenamiento acaba de terminar y el objeto sigue en memoria RAM
        if hasattr(modelo, 'trainer') and modelo.trainer and hasattr(modelo.trainer, 'optimizer'):
            # El optimizador es un objeto (ej. <torch.optim.sgd.SGD object at 0x...>)
            # Obtenemos su nombre de clase real
            opt_obj = modelo.trainer.optimizer
            nombre_real = type(opt_obj).__name__
            
            # También podemos sacar el Learning Rate real final
            lr_final = opt_obj.param_groups[0]['lr']
            
            print(f"Decisión de 'Auto':")
            print(f"   • Optimizador:   {nombre_real}") # Dirá 'SGD' o 'AdamW'
            print(f"   • Learning Rate: {lr_final:.6f}")
            return

        # 2. Si el modelo fue cargado desde disco (.pt) y no hay trainer en memoria
        # Buscamos en los metadatos internos del archivo
        if hasattr(modelo, 'ckpt') and modelo.ckpt:
            train_args = modelo.ckpt.get('train_args', {})
            # A veces aquí también dice 'auto', en cuyo caso la única verdad está en los logs de texto
            print(f"Configuración guardada: {train_args.get('optimizer', 'Desconocido')}")
            print("Si aquí dice 'auto', por favor revisa el archivo '/runs/.../train/main.log'")

    except Exception as e:
        print(f"No se pudo recuperar automáticamente: {e}")

In [3]:
# Entrenamiento YOLOv11n con dataset Resplit
run_name_v11n_resplit_tl = f"103_v11n_resplit_tl"

print(f"Iniciando entrenamiento: {run_name_v11n_resplit_tl}")

# Cargar modelo Nano pre-entrenado
model_v11n_resplit_tl = YOLO('yolo11n.pt')

results_v11n_resplit_tl = model_v11n_resplit_tl.train(
    data=YAML_CLEAN,  # Dataset resplit
    model='yolo11n.pt',  # Modelo pre-entrenado
    project=MODELS_DIR,
    name=run_name_v11n_resplit_tl,

    epochs=300,            # Ajustable
    patience=50,          # Early stopping    
    batch=32,          # Ajustable
    imgsz=640,            # Tamaño de imagen    
    
    exist_ok=False,         # Sobrescribir si existe
    pretrained=True,
    optimizer='AdamW',
    verbose=True,
    workers=os.cpu_count(),
    close_mosaic=10 # Apaga aumentación durante las últimas 10 épocas.
)

Iniciando entrenamiento: 103_v11n_resplit_tl
New https://pypi.org/project/ultralytics/8.4.128 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.116 🚀 Python-3.10.20 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4070 Ti, 11873MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=../../datasets/02_placas/data.yaml, degrees=0.0, deterministic=True, device=, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=300, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, keras=False, kobj=1.0, line_width=None, 

/home/robertoplr/miniconda3/envs/proyecto/lib/python3.10/site-packages/ultralytics/utils/tal.py:401: UserWarning: cumsum_cuda_kernel does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at ../aten/src/ATen/Context.cpp:91.)
  sx = feats[0].new_full((w,), 1, dtype=dtype).cumsum(0) - (1 - grid_cell_offset)  # shift x
/home/robertoplr/miniconda3/envs/proyecto/lib/python3.10/site-packages/ultralytics/utils/tal.py:402: UserWarning: cumsum_cuda_kernel does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at ../aten/src/ATen/Context.cpp:91.)
  sy = feat

AMP: checks passed ✅
train: Fast image access ✅ (ping: 0.0±0.0 ms, read: 10883.0±1749.4 MB/s, size: 1945.8 KB)
train: Scanning /home/robertoplr/Documentos/moca_proyecto/datasets/02_placas/train/labels... 794 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 794/794 3.4Kit/s 0.2s0.1ss
train: New cache created: /home/robertoplr/Documentos/moca_proyecto/datasets/02_placas/train/labels.cache
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3200.6±2592.1 MB/s, size: 2155.4 KB)
val: Scanning /home/robertoplr/Documentos/moca_proyecto/datasets/02_placas/val/labels... 99 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 99/99 2.2Kit/s 0.0s
val: New cache created: /home/robertoplr/Documentos/moca_proyecto/datasets/02_placas/val/labels.cache
optimizer: AdamW(lr=0.01, momentum=0.937) with parameter groups 81 weight(decay=0.0), 88 weight(decay=0.0005), 87 bias(decay=0.0)
Plotting labels to /home/robertoplr/Documentos/moca_proyecto/models/02_placas/103_v11n_resplit_tl/labels.jpg... 
Image sizes

/home/robertoplr/miniconda3/envs/proyecto/lib/python3.10/site-packages/ultralytics/utils/tal.py:401: UserWarning: cumsum_cuda_kernel does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at ../aten/src/ATen/Context.cpp:91.)
  sx = feats[0].new_full((w,), 1, dtype=dtype).cumsum(0) - (1 - grid_cell_offset)  # shift x
/home/robertoplr/miniconda3/envs/proyecto/lib/python3.10/site-packages/ultralytics/utils/tal.py:402: UserWarning: cumsum_cuda_kernel does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at ../aten/src/ATen/Context.cpp:91.)
  sy = feat


      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      1/300      4.36G      2.078      3.304      1.397         60        640: 100% ━━━━━━━━━━━━ 25/25 1.5it/s 16.5s0.3s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 1.8it/s 1.1s2.3s
                   all         99        115          0          0          0          0

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      2/300      4.69G      1.836      1.561      1.391         49        640: 0% ──────────── 0/25  1.4s

...

    298/300       4.7G     0.4182     0.2468     0.8017         29        640: 100% ━━━━━━━━━━━━ 25/25 5.6it/s 4.5s0.2s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 3.5s/it 7.0s<23.1s
                   all         99        115      0.998      0.983      0.992      0.879

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
    299/300       4.7G     0.4778     0.2619     0.8222         39        640: 4% ──────────── 1/25 1.4it/s 0.2s<16.8s

/home/robertoplr/miniconda3/envs/proyecto/lib/python3.10/site-packages/ultralytics/utils/tal.py:401: UserWarning: cumsum_cuda_kernel does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at ../aten/src/ATen/Context.cpp:91.)
  sx = feats[0].new_full((w,), 1, dtype=dtype).cumsum(0) - (1 - grid_cell_offset)  # shift x
/home/robertoplr/miniconda3/envs/proyecto/lib/python3.10/site-packages/ultralytics/utils/tal.py:402: UserWarning: cumsum_cuda_kernel does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at ../aten/src/ATen/Context.cpp:91.)
  sy = feat

    299/300       4.7G     0.4115     0.2471     0.7975         29        640: 100% ━━━━━━━━━━━━ 25/25 7.0it/s 3.6s0.2s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 3.9s/it 7.7s<25.4s
                   all         99        115      0.999      0.983      0.993      0.879

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
    300/300       4.7G     0.4251     0.2364      0.812         36        640: 4% ──────────── 1/25 3.0it/s 0.2s<8.1s

/home/robertoplr/miniconda3/envs/proyecto/lib/python3.10/site-packages/ultralytics/utils/tal.py:401: UserWarning: cumsum_cuda_kernel does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at ../aten/src/ATen/Context.cpp:91.)
  sx = feats[0].new_full((w,), 1, dtype=dtype).cumsum(0) - (1 - grid_cell_offset)  # shift x
/home/robertoplr/miniconda3/envs/proyecto/lib/python3.10/site-packages/ultralytics/utils/tal.py:402: UserWarning: cumsum_cuda_kernel does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at ../aten/src/ATen/Context.cpp:91.)
  sy = feat

    300/300       4.7G     0.4158     0.2455     0.8004         28        640: 100% ━━━━━━━━━━━━ 25/25 6.6it/s 3.8s0.2s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 2/2 4.1s/it 8.1s<26.8s
                   all         99        115      0.999      0.983      0.993      0.882

300 epochs completed in 1.018 hours.
Optimizer stripped from /home/robertoplr/Documentos/moca_proyecto/models/02_placas/103_v11n_resplit_tl/weights/last.pt, 5.5MB
Optimizer stripped from /home/robertoplr/Documentos/moca_proyecto/models/02_placas/103_v11n_resplit_tl/weights/best.pt, 5.5MB

Validating /home/robertoplr/Documentos/moca_proyecto/models/02_placas/103_v11n_resplit_tl/weights/best.pt...
Ultralytics 8.4.116 🚀 Python-3.10.20 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4070 Ti, 11873MiB)
YOLO11n summary (fused): 101 layers, 2,582,347 parameters, 0 gradients, 6.4 GFLOPs
                 Class     Images  Instances      Box(P          R      mAP5

In [3]:
# Entrenamiento YOLOv11n con dataset Resplit
run_name_v11n_resplit_tl = f"104_v11n_resplit_tl"

print(f"Iniciando entrenamiento: {run_name_v11n_resplit_tl}")

# Cargar modelo Nano pre-entrenado
model_v11n_resplit_tl = YOLO('yolo11n.pt')

results_v11n_resplit_tl = model_v11n_resplit_tl.train(
    data=YAML_CLEAN,  # Dataset resplit
    model='yolo11n.pt',  # Modelo pre-entrenado
    project=MODELS_DIR,
    name=run_name_v11n_resplit_tl,

    epochs=300,            # Ajustable
    patience=50,          # Early stopping    
    batch=16,          # Ajustable
    imgsz=800,            # Tamaño de imagen    
    device=0,      # Ajustable según disponibilidad de GPU
    box=10.0,
    dfl=2.0,
    exist_ok=False,         # Sobrescribir si existe
    pretrained=True,
    optimizer='AdamW',
    cos_lr=True,
    verbose=True,
    workers=os.cpu_count(),
    close_mosaic=10 # Apaga aumentación durante las últimas 20 épocas.
)

Iniciando entrenamiento: 104_v11n_resplit_tl
New https://pypi.org/project/ultralytics/8.4.129 available 😃 Update with 'pip install -U ultralytics'
Ultralytics 8.4.116 🚀 Python-3.10.20 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4070 Ti, 11876MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=10.0, cache=False, cfg=None, channels_last=False, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=../../datasets/02_placas/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=2.0, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=300, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=800, iou=0.7, keras=False, kobj=1.0, line_width=None,

/home/robertoplr/miniconda3/envs/proyecto/lib/python3.10/site-packages/ultralytics/utils/tal.py:401: UserWarning: cumsum_cuda_kernel does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at ../aten/src/ATen/Context.cpp:91.)
  sx = feats[0].new_full((w,), 1, dtype=dtype).cumsum(0) - (1 - grid_cell_offset)  # shift x
/home/robertoplr/miniconda3/envs/proyecto/lib/python3.10/site-packages/ultralytics/utils/tal.py:402: UserWarning: cumsum_cuda_kernel does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at ../aten/src/ATen/Context.cpp:91.)
  sy = feat

AMP: checks passed ✅
train: Fast image access ✅ (ping: 0.0±0.0 ms, read: 715.8±83.6 MB/s, size: 1945.8 KB)
train: Scanning /home/robertoplr/Documentos/moca_proyecto/datasets/02_placas/train/labels.cache... 794 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 794/794 185.0Mit/s 0.0s
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 582.1±206.2 MB/s, size: 2155.4 KB)
val: Scanning /home/robertoplr/Documentos/moca_proyecto/datasets/02_placas/val/labels.cache... 99 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 99/99 4.2Mit/s 0.0s
optimizer: AdamW(lr=0.01, momentum=0.937) with parameter groups 81 weight(decay=0.0), 88 weight(decay=0.0005), 87 bias(decay=0.0)
Plotting labels to /home/robertoplr/Documentos/moca_proyecto/models/02_placas/104_v11n_resplit_tl/labels.jpg... 
Image sizes 800 train, 800 val
Using 12 dataloader workers
Logging results to /home/robertoplr/Documentos/moca_proyecto/models/02_placas/104_v11n_resplit_tl
Starting training for 300 epochs...


/home/robertoplr/miniconda3/envs/proyecto/lib/python3.10/site-packages/ultralytics/utils/tal.py:401: UserWarning: cumsum_cuda_kernel does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at ../aten/src/ATen/Context.cpp:91.)
  sx = feats[0].new_full((w,), 1, dtype=dtype).cumsum(0) - (1 - grid_cell_offset)  # shift x
/home/robertoplr/miniconda3/envs/proyecto/lib/python3.10/site-packages/ultralytics/utils/tal.py:402: UserWarning: cumsum_cuda_kernel does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at ../aten/src/ATen/Context.cpp:91.)
  sy = feat


      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
      1/300      3.45G      2.244      2.495      1.718         20        800: 100% ━━━━━━━━━━━━ 50/50 2.3it/s 21.5s0.4s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 3.6it/s 1.1s0.3s
                   all         99        115          0          0          0          0

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


...

    275/300      3.94G     0.5601     0.2591      1.092         18        800: 100% ━━━━━━━━━━━━ 50/50 3.6it/s 13.8s0.3s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 4.7it/s 0.8s0.4s
                   all         99        115          1      0.989      0.995      0.908

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
    276/300      3.94G     0.5442     0.2287      1.056         35        800: 0% ──────────── 0/50  0.3s

/home/robertoplr/miniconda3/envs/proyecto/lib/python3.10/site-packages/ultralytics/utils/tal.py:401: UserWarning: cumsum_cuda_kernel does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at ../aten/src/ATen/Context.cpp:91.)
  sx = feats[0].new_full((w,), 1, dtype=dtype).cumsum(0) - (1 - grid_cell_offset)  # shift x
/home/robertoplr/miniconda3/envs/proyecto/lib/python3.10/site-packages/ultralytics/utils/tal.py:402: UserWarning: cumsum_cuda_kernel does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at ../aten/src/ATen/Context.cpp:91.)
  sy = feat

    276/300      3.94G     0.5342     0.2479      1.081         21        800: 100% ━━━━━━━━━━━━ 50/50 3.3it/s 15.1s0.2s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 7.6it/s 0.5s0.2s
                   all         99        115          1      0.983      0.995      0.903

      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size
    277/300      3.94G     0.6811     0.3311      1.098         36        800: 0% ──────────── 0/50  0.2s

/home/robertoplr/miniconda3/envs/proyecto/lib/python3.10/site-packages/ultralytics/utils/tal.py:401: UserWarning: cumsum_cuda_kernel does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at ../aten/src/ATen/Context.cpp:91.)
  sx = feats[0].new_full((w,), 1, dtype=dtype).cumsum(0) - (1 - grid_cell_offset)  # shift x
/home/robertoplr/miniconda3/envs/proyecto/lib/python3.10/site-packages/ultralytics/utils/tal.py:402: UserWarning: cumsum_cuda_kernel does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at ../aten/src/ATen/Context.cpp:91.)
  sy = feat

    277/300      3.94G     0.5578      0.258      1.085         25        800: 100% ━━━━━━━━━━━━ 50/50 4.2it/s 11.9s0.2s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 4/4 5.2it/s 0.8s0.3s
                   all         99        115          1      0.991      0.995      0.908
EarlyStopping: Training stopped early as no improvement observed in last 50 epochs. Best results observed at epoch 227, best model saved as best.pt.
To update EarlyStopping(patience=50) pass a new patience value, i.e. `patience=300` or use `patience=0` to disable EarlyStopping.

277 epochs completed in 1.067 hours.
Optimizer stripped from /home/robertoplr/Documentos/moca_proyecto/models/02_placas/104_v11n_resplit_tl/weights/last.pt, 5.5MB
Optimizer stripped from /home/robertoplr/Documentos/moca_proyecto/models/02_placas/104_v11n_resplit_tl/weights/best.pt, 5.5MB

Validating /home/robertoplr/Documentos/moca_proyecto/models/02_placas/104_v11n_resplit_tl/w

In [4]:
# --- RECUPERAR HIPERPARÁMETROS REALES ---
# Ejecutar al finalizar el entrenamiento
print(f"Batch Size: {model_v11n_resplit_tl.trainer.args.batch}") 
print(f"Optimizer: {obtener_optimizador_real(model_v11n_resplit_tl)}")
print(f"Learning Rate inicial: {model_v11n_resplit_tl.trainer.args.lr0}")

Batch Size: 16
Decisión de 'Auto':
   • Optimizador:   AdamW
   • Learning Rate: 0.000256
Optimizer: None
Learning Rate inicial: 0.01


In [11]:
print("Validando modelo 104_v11n_resplit_tl en split='test'...")

PROJECT_DIR_REL = '../../models/02_placas'
PROJECT_DIR = os.path.abspath(PROJECT_DIR_REL)
run_name = f"104_v11n_resplit_tl"

# Cargar el MEJOR modelo resultante del entrenamiento anterior
model_yolov8n_baseline_tl = os.path.join(PROJECT_DIR, run_name, 'weights', 'best.pt')
best_model = YOLO(model_yolov8n_baseline_tl)

# Ejecutar validación en split='test'
metrics = best_model.val(
    split='test',
    project=PROJECT_DIR, 
    name=f"{run_name}_eval", 
    conf=0.5,          # Umbral de confianza,
    imgsz=640,      # Tamaño de imagen
    batch=49,        # Mismo batch que entrenamiento
    plots=True       # Generar gráficos de métricas
)

print(f"\nResultados Finales en Test del dataset Baseline:")
print(f"   mAP@50:    {metrics.box.map50:.4f} (Precisión holgada)")
print(f"   mAP@50-95: {metrics.box.map:.4f}  (Precisión estricta )")

Validando modelo 104_v11n_resplit_tl en split='test'...
Ultralytics 8.4.116 🚀 Python-3.10.20 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4070 Ti, 11876MiB)
YOLO11n summary (fused): 101 layers, 2,582,347 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 13822.5±1879.8 MB/s, size: 2705.7 KB)
val: Scanning /home/robertoplr/Documentos/moca_proyecto/datasets/02_placas/test/labels.cache... 100 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 100/100 30.0Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 3/3 2.0s/it 5.9s1.4s4s
                   all        100        123      0.992      0.976      0.975      0.861
Speed: 3.0ms preprocess, 1.3ms inference, 0.0ms loss, 0.4ms postprocess per image
Results saved to /home/robertoplr/Documentos/moca_proyecto/models/02_placas/104_v11n_resplit_tl_eval

Resultados Finales en Test del dataset Baseline:
   mAP@50:    0.9749 (Precisión holgada)
   

In [12]:
import glob
import random

run_name = f"103_v11n_resplit_tl_inference"

# Tomar una imagen de prueba aleatoria
test_images = glob.glob('../../datasets/02_placas/test/images/*.jpg')
if test_images:
    sample_img = random.choice(test_images)
    
    # Prediccion
    res = model_v11n_resplit_tl.predict(sample_img, save=True, project=MODELS_DIR, name=run_name)
     
    print(f"Inferencia guardada en {res[0].save_dir}")
else:
    print("No se encontraron imagenes de prueba.")


image 1/1 /home/robertoplr/Documentos/moca_proyecto/notebooks/02_placas/../../datasets/02_placas/test/images/01921.jpg: 608x800 2 license_plates, 5.8ms
Speed: 2.1ms preprocess, 5.8ms inference, 1.0ms postprocess per image at shape (1, 3, 608, 800)
Results saved to /home/robertoplr/Documentos/moca_proyecto/models/02_placas/103_v11n_resplit_tl_inference
Inferencia guardada en /home/robertoplr/Documentos/moca_proyecto/models/02_placas/103_v11n_resplit_tl_inference


In [13]:
# MODELS_DIR = '../../models/02_placas'
# Cargar mejores pesos
# Asegúrate que estos nombres coincidan exactamente con los definidos en las celdas de entrenamiento
run_name_v11 = f"104_v11n_resplit_tl"

PROJECT_DIR_REL = '../../models/02_placas'
PROJECT_DIR = os.path.abspath(PROJECT_DIR_REL)

path_v11_weights = os.path.join(PROJECT_DIR, run_name_v11, 'weights', 'best.pt')

model_final_v11 = YOLO(path_v11_weights)

print("--- VALIDACION CRUZADA ---")

# 1. Validar Modelo YOLOv11n
metrics_v11 = model_final_v11.val(
    data=YAML_CLEAN, 
    split='test', 
    project=PROJECT_DIR,
    name=f"{run_name_v11}_val2"
)

print("\nRESULTADOS COMPARATIVOS (mAP50-95):")
print(f"Modelo YOLOv11n: {metrics_v11.box.map:.4f}")

--- VALIDACION CRUZADA ---
Ultralytics 8.4.116 🚀 Python-3.10.20 torch-2.5.1+cu121 CUDA:0 (NVIDIA GeForce RTX 4070 Ti, 11876MiB)
YOLO11n summary (fused): 101 layers, 2,582,347 parameters, 0 gradients, 6.4 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 9806.0±4506.4 MB/s, size: 2566.2 KB)
val: Scanning /home/robertoplr/Documentos/moca_proyecto/datasets/02_placas/test/labels.cache... 100 images, 0 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 100/100 32.3Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 7/7 2.2it/s 3.1s0.2ss
                   all        100        123      0.991      0.976      0.994      0.886
Speed: 1.7ms preprocess, 2.7ms inference, 0.0ms loss, 0.8ms postprocess per image
Results saved to /home/robertoplr/Documentos/moca_proyecto/models/02_placas/104_v11n_resplit_tl_val2

RESULTADOS COMPARATIVOS (mAP50-95):
Modelo YOLOv11n: 0.8863


In [14]:
print("\nINFORMACION DEL MODELO YOLOv11n:")
# print(model_final_v11.info)
model_final_v11.info()


INFORMACION DEL MODELO YOLOv11n:
YOLO11n summary (fused): 101 layers, 2,582,347 parameters, 0 gradients, 6.4 GFLOPs


(101, 2582347, 0, 6.3719936)

In [15]:
from torchinfo import summary

def desplegar_arquitectura_completa(modelo_yolo, input_size=(1, 3, 640, 640)):
    """
    Muestra el resumen completo de capas, params y tamaños de memoria.
    input_size: (Batch, Canales, Alto, Ancho)
    """
    print(f"\n🔍 ARQUITECTURA DETALLADA: {modelo_yolo.task_map}")
    # Accedemos al modelo interno de PyTorch (modelo.model)
    summary(modelo_yolo.model, 
            input_size=input_size, 
            col_names=["input_size", "output_size", "num_params", "kernel_size", "mult_adds"],
            verbose=1)

In [16]:
desplegar_arquitectura_completa(model_final_v11, input_size=(1, 3, 640, 640))


🔍 ARQUITECTURA DETALLADA: {'classify': {'model': <class 'ultralytics.nn.tasks.ClassificationModel'>, 'trainer': <class 'ultralytics.models.yolo.classify.train.ClassificationTrainer'>, 'validator': <class 'ultralytics.models.yolo.classify.val.ClassificationValidator'>, 'predictor': <class 'ultralytics.models.yolo.classify.predict.ClassificationPredictor'>}, 'detect': {'model': <class 'ultralytics.nn.tasks.DetectionModel'>, 'trainer': <class 'ultralytics.models.yolo.detect.train.DetectionTrainer'>, 'validator': <class 'ultralytics.models.yolo.detect.val.DetectionValidator'>, 'predictor': <class 'ultralytics.models.yolo.detect.predict.DetectionPredictor'>}, 'segment': {'model': <class 'ultralytics.nn.tasks.SegmentationModel'>, 'trainer': <class 'ultralytics.models.yolo.segment.train.SegmentationTrainer'>, 'validator': <class 'ultralytics.models.yolo.segment.val.SegmentationValidator'>, 'predictor': <class 'ultralytics.models.yolo.segment.predict.SegmentationPredictor'>}, 'pose': {'model'